# 02 - Load PrimeKG into Neo4j

This notebook loads the prepared staging files into Neo4j. It creates constraints, indexes, enriched nodes, and deduplicated undirected relationships.

Run `01_prepare_primekg_for_neo4j.ipynb` before this notebook.

## 1. Connection Configuration

Configure Neo4j connection settings through environment variables when possible. A local Neo4j instance can use the defaults shown below.

In [1]:
from pathlib import Path
import math
import os
import re

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
from tqdm.auto import tqdm

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

STAGING_DIR = Path("build/neo4j_import")
NODES_CSV = STAGING_DIR / "nodes_enriched.csv"
RELATIONSHIPS_CSV = STAGING_DIR / "relationships_undirected.csv"
RELATION_TYPE_MAP_CSV = STAGING_DIR / "relation_type_map.csv"

print(f"Neo4j URI: {NEO4J_URI}")
print(f"Neo4j database: {NEO4J_DATABASE}")
# print(f"Staging directory: {STAGING_DIR.resolve()}")

Neo4j URI: neo4j://127.0.0.1:7687
Neo4j database: neo4j


## 2. Connect to Neo4j

Create the Neo4j driver and verify connectivity before modifying the database.

In [2]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j")

Connected to Neo4j


In [3]:
with driver.session(database="system") as session:
    result = session.run("SHOW DATABASES")
    for record in result:
        print(record.data())

{'name': 'neo4j', 'type': 'standard', 'aliases': [], 'access': 'read-write', 'address': 'localhost:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'default': True, 'home': True, 'constituents': []}
{'name': 'system', 'type': 'system', 'aliases': [], 'access': 'read-write', 'address': 'localhost:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'default': False, 'home': False, 'constituents': []}


## 3. Optional Database Reset

Set `RESET_DATABASE = True` only when you intentionally want to delete all nodes and relationships from the configured Neo4j database. Leave it as `False` for normal incremental development.

In [4]:
RESET_DATABASE = False

def run_write(query, parameters=None):
    with driver.session(database=NEO4J_DATABASE) as session:
        return session.execute_write(lambda tx: list(tx.run(query, parameters or {})))

if RESET_DATABASE:
    run_write("MATCH (n) DETACH DELETE n")
    print("Database cleared")
else:
    print("Database reset skipped")

Database reset skipped


## 4. Create Constraints and Indexes

Create constraints and indexes used by the backend API. These support stable lookup by PrimeKG index, name search, type filters, and relationship filtering.

In [5]:
schema_statements = [
    "CREATE CONSTRAINT entity_primekg_index IF NOT EXISTS FOR (n:Entity) REQUIRE n.primekg_index IS UNIQUE",
    "CREATE INDEX entity_source_node_id IF NOT EXISTS FOR (n:Entity) ON (n.source_node_id)",
    "CREATE INDEX entity_name_lc IF NOT EXISTS FOR (n:Entity) ON (n.name_lc)",
    "CREATE INDEX entity_node_type IF NOT EXISTS FOR (n:Entity) ON (n.node_type)",
    "CREATE INDEX entity_node_source IF NOT EXISTS FOR (n:Entity) ON (n.node_source)",
    "CREATE INDEX disease_mondo_id IF NOT EXISTS FOR (n:Disease) ON (n.disease_mondo_id)",
    "CREATE INDEX disease_group_name_bert IF NOT EXISTS FOR (n:Disease) ON (n.disease_group_name_bert)",
    "CREATE FULLTEXT INDEX entity_fulltext IF NOT EXISTS FOR (n:Entity) ON EACH [n.name, n.name_lc, n.search_text]",
]

for statement in schema_statements:
    run_write(statement)

# Neo4j relationship property indexes must target a concrete relationship type.
def safe_schema_identifier(value):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", str(value)):
        raise ValueError(f"Unsafe Cypher identifier: {value!r}")
    return str(value)

relation_type_map_for_schema = pd.read_csv(RELATION_TYPE_MAP_CSV)
relationship_schema_statements = []
for rel_type in sorted(relation_type_map_for_schema["neo4j_relationship_type"].dropna().unique()):
    safe_rel_type = safe_schema_identifier(rel_type)
    rel_type_lc = safe_rel_type.lower()
    relationship_schema_statements.extend(
        [
            f"CREATE INDEX rel_{rel_type_lc}_relation IF NOT EXISTS FOR ()-[r:{safe_rel_type}]-() ON (r.relation)",
            f"CREATE INDEX rel_{rel_type_lc}_display_relation IF NOT EXISTS FOR ()-[r:{safe_rel_type}]-() ON (r.display_relation)",
        ]
    )

for statement in relationship_schema_statements:
    run_write(statement)

print(
    f"Created or verified {len(schema_statements)} node schema objects "
    f"and {len(relationship_schema_statements)} relationship schema objects"
)

Created or verified 8 node schema objects and 60 relationship schema objects


## 5. Utility Functions for Batched Loading

Define helpers for cleaning pandas rows and safely injecting known labels/relationship types into Cypher. Data values remain parameterized.

In [6]:
def sanitize_cypher_identifier(value: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Unsafe Cypher identifier: {value!r}")
    return value

def clean_value(value):
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    return value

def dataframe_to_records(df: pd.DataFrame):
    records = []
    for row in df.to_dict(orient="records"):
        cleaned = {key: clean_value(value) for key, value in row.items()}
        records.append(cleaned)
    return records

def batched_count(path: Path):
    return sum(1 for _ in path.open(encoding="utf-8")) - 1

## 6. Load Enriched Nodes

Load all nodes with the generic labels `Entity` and `PrimeKG`, plus a type-specific label such as `Disease`, `Drug`, or `GeneProtein`. This keeps general traversal simple while still allowing label-specific queries.

In [7]:
NODE_BATCH_SIZE = 5_000
node_total = batched_count(NODES_CSV)

node_query_template = """
UNWIND $rows AS row
MERGE (n:Entity:PrimeKG:{label} {{primekg_index: row.primekg_index}})
SET n += row
"""

with tqdm(total=node_total, desc="Loading nodes") as progress:
    for chunk in pd.read_csv(NODES_CSV, chunksize=NODE_BATCH_SIZE):
        chunk["primekg_index"] = chunk["primekg_index"].astype(int)
        for label, label_df in chunk.groupby("app_label"):
            safe_label = sanitize_cypher_identifier(label)
            query = node_query_template.format(label=safe_label)
            rows = dataframe_to_records(label_df)
            run_write(query, {"rows": rows})
        progress.update(len(chunk))

print(f"Loaded or updated {node_total:,} nodes")

Loading nodes:   0%|          | 0/129484 [00:00<?, ?it/s]

Loaded or updated 129,484 nodes


## 7. Load Deduplicated Relationships

Load each relationship once using an undirected `MERGE`. Relationship type names are generated from the relation map, and the original PrimeKG relation labels remain stored as properties.

In [9]:
REL_BATCH_SIZE = 20_000
relation_type_map = pd.read_csv(RELATION_TYPE_MAP_CSV)
relation_to_type = dict(zip(relation_type_map["relation"], relation_type_map["neo4j_relationship_type"]))
relationship_total = batched_count(RELATIONSHIPS_CSV)

relationship_query_template = """
UNWIND $rows AS row
MATCH (source:Entity {{primekg_index: row.source_index}})
MATCH (target:Entity {{primekg_index: row.target_index}})
MERGE (source)-[r:{rel_type}]-(target)
SET r.relation = row.relation,
    r.display_relation = row.display_relation,
    r.primekg_row_count = row.primekg_row_count,
    r.source = 'PrimeKG',
    r.undirected = true
"""

with tqdm(total=relationship_total, desc="Loading relationships") as progress:
    for chunk in pd.read_csv(RELATIONSHIPS_CSV, chunksize=REL_BATCH_SIZE):
        chunk["source_index"] = chunk["source_index"].astype(int)
        chunk["target_index"] = chunk["target_index"].astype(int)
        chunk["primekg_row_count"] = chunk["primekg_row_count"].astype(int)
        chunk["neo4j_relationship_type"] = chunk["relation"].map(relation_to_type)

        if chunk["neo4j_relationship_type"].isna().any():
            missing = chunk.loc[chunk["neo4j_relationship_type"].isna(), "relation"].unique()
            raise ValueError(f"Missing relationship type mappings: {missing}")

        for rel_type, rel_df in chunk.groupby("neo4j_relationship_type"):
            safe_rel_type = sanitize_cypher_identifier(rel_type)
            query = relationship_query_template.format(rel_type=safe_rel_type)
            rows = dataframe_to_records(
                rel_df[["source_index", "target_index", "relation", "display_relation", "primekg_row_count"]]
            )
            run_write(query, {"rows": rows})

        progress.update(len(chunk))

print(f"Loaded or updated {relationship_total:,} undirected relationships")

Loading relationships:   0%|          | 0/4050249 [00:00<?, ?it/s]

Loaded or updated 4,050,249 undirected relationships


## 8. Basic Post-Load Counts

Run basic database counts to confirm that nodes and relationships are present after loading.

In [10]:
with driver.session(database=NEO4J_DATABASE) as session:
    node_count = session.run("MATCH (n:Entity) RETURN count(n) AS count").single()["count"]
    rel_count = session.run("MATCH ()-[r]-() RETURN count(r) AS count").single()["count"]

print(f"Entity nodes: {node_count:,}")
print(f"Relationship traversals counted by Cypher: {rel_count:,}")
print("Note: MATCH ()-[r]-() counts each stored relationship once in Neo4j.")

Entity nodes: 129,375
Relationship traversals counted by Cypher: 8,100,128
Note: MATCH ()-[r]-() counts each stored relationship once in Neo4j.


## 9. Close the Driver

Close the Neo4j driver when ingestion is complete.

In [11]:
driver.close()
print("Neo4j driver closed")

Neo4j driver closed
